# Tinh chỉnh Deep Learning Models (LSTM & GRU)

Notebook này tìm ra kiến trúc mạnh mẽ nhất cho các mạng Neural Networks để bắt được các mẫu phi tuyến tính phức tạp trong dữ liệu PM2.5.
Các bước thực hiện:
1. **Kiểm soát Overfitting & Tối ưu hóa:** Tích hợp `ReduceLROnPlateau` và `EarlyStopping`. Thiết lập tham số Dropout.
2. **Architecture Grid Search:** Lặp qua các không gian tìm kiếm:
    - Loại RNN: `LSTM` vs `GRU`
    - Chiều kích Hidden State: `64`, `128`, `256`
    - Số lượng Layer (Depth): `1`, `2`
    - Direction: `Bidirectional` (Học cả 2 chiều dữ liệu) vs `Unidirectional`.
3. **Đánh giá và Lưu kết quả:** Tìm ra tập tham số trả về RMSE thấp nhất trên tập Validation. 


In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time
import pickle
import itertools

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_squared_error, mean_absolute_error

sns.set_theme(style="whitegrid", context="notebook")

FIG_DIR = Path("../outputs/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR = Path("../outputs/predictions")
PRED_DIR.mkdir(parents=True, exist_ok=True)
MODELING_DIR = Path("../data/processed/modeling_fs")
TARGET_LOG = "target_pm25_h24_log"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Sử dụng phần cứng:", device)


Sử dụng phần cứng: cpu


In [2]:
def regression_metrics(y_true, y_pred, inverse=True):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    
    if inverse:
        y_true_inv = np.expm1(y_true)
        y_pred_inv = np.expm1(y_pred)
    else:
        y_true_inv = y_true
        y_pred_inv = y_pred
        
    rmse = float(np.sqrt(mean_squared_error(y_true_inv, y_pred_inv)))
    mae = mean_absolute_error(y_true_inv, y_pred_inv)
    eps = 1e-6
    mask = np.abs(y_true_inv) > eps
    mape = np.mean(np.abs((y_true_inv[mask] - y_pred_inv[mask]) / y_true_inv[mask])) * 100.0
    return {"RMSE": rmse, "MAE": mae, "MAPE": mape}

def load_xy(path: Path):
    df = pd.read_csv(path, parse_dates=["datetime_local"])
    feat = [c for c in df.columns if c not in ("datetime_local", TARGET_LOG)]
    return df[feat], df[TARGET_LOG], df["datetime_local"]

train_X, train_y, train_dt = load_xy(MODELING_DIR / "train_dl.csv")
val_X, val_y, val_dt = load_xy(MODELING_DIR / "val_dl.csv")
test_X, test_y, test_dt = load_xy(MODELING_DIR / "test_dl.csv")
num_features = train_X.shape[1]

# Reshape cho RNN (batch, seq_len=1, features)
def create_dataloaders(batch_size=64):
    X_tr_t = torch.tensor(train_X.values, dtype=torch.float32).unsqueeze(1)
    y_tr_t = torch.tensor(train_y.values, dtype=torch.float32).unsqueeze(1)
    
    X_v_t = torch.tensor(val_X.values, dtype=torch.float32).unsqueeze(1)
    y_v_t = torch.tensor(val_y.values, dtype=torch.float32).unsqueeze(1)
    
    X_te_t = torch.tensor(test_X.values, dtype=torch.float32).unsqueeze(1)
    y_te_t = torch.tensor(test_y.values, dtype=torch.float32).unsqueeze(1)
    
    train_dl = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=batch_size, shuffle=False)
    val_dl = DataLoader(TensorDataset(X_v_t, y_v_t), batch_size=batch_size, shuffle=False)
    test_dl = DataLoader(TensorDataset(X_te_t, y_te_t), batch_size=batch_size, shuffle=False)
    
    return train_dl, val_dl, test_dl, X_tr_t, X_v_t, X_te_t

train_loader, val_loader, test_loader, X_tr_t, X_v_t, X_te_t = create_dataloaders()


## 1. Xây dựng Kiến trúc Deep RNN & Các Plugin Trợ trợ
Bao gồm một class Network hỗ trợ linh hoạt các tham số Tuning và công cụ `EarlyStopper`.

In [3]:
class DynamicRNN(nn.Module):
    def __init__(self, rnn_type="LSTM", input_size=10, hidden_size=64, num_layers=1, dropout=0.0, bidirectional=False):
        super(DynamicRNN, self).__init__()
        self.rnn_type = rnn_type
        
        # RNN Layer
        if rnn_type == "LSTM":
            self.rnn = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, 
                               dropout=dropout if num_layers > 1 else 0, 
                               bidirectional=bidirectional)
        else:
            self.rnn = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, 
                              dropout=dropout if num_layers > 1 else 0, 
                              bidirectional=bidirectional)
            
        # MLP Decoder
        fc_dim = hidden_size * 2 if bidirectional else hidden_size
        self.fc = nn.Sequential(
            nn.Linear(fc_dim, fc_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fc_dim // 2, 1)
        )

    def forward(self, x):
        out, _ = self.rnn(x)
        out = out[:, -1, :] # Trích xuất time-step cuối cùng
        return self.fc(out)

class EarlyStopper:
    def __init__(self, patience=5, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.min_validation_loss = float('inf')

    def early_stop(self, validation_loss):
        if validation_loss < (self.min_validation_loss - self.min_delta):
            self.min_validation_loss = validation_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False


## 2. Thiết lập Vòng lặp Học (Training Loop) với Scheduler

In [7]:
def train_eval_model(model, train_loader, val_loader, epochs=50, lr=1e-3, patience=10, verbose=False):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5) # L2 Regularization chống quá khớp
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)
    early_stopper = EarlyStopper(patience=patience)
    
    model.to(device)
    best_val_loss = float('inf')
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            preds = model(X_b)
            loss = criterion(preds, y_b)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X_b.size(0)
            
        train_loss /= len(train_loader.dataset)
        
        # Đánh giá Val
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_b, y_b in val_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                preds = model(X_b)
                loss = criterion(preds, y_b)
                val_loss += loss.item() * X_b.size(0)
        val_loss /= len(val_loader.dataset)
        
        # Cập nhật LR
        scheduler.step(val_loss)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            
        if early_stopper.early_stop(val_loss):
            if verbose:
                print(f"Early Stopping at Epoch {epoch+1}. Best Val Loss: {best_val_loss:.5f}")
            break
            
    return best_val_loss

def predict(model, X_tensor):
    model.eval()
    with torch.no_grad():
         preds = model(X_tensor.to(device)).cpu().numpy().flatten()
    return preds


## 3. Architecture Grid Search

Thiết lập không gian các siêu tham số (Hyperparameters) cần tối ưu. Máy tính sẽ thử nghiệm mỗi sự kết hợp (Combination) và xếp hạng chúng dựa trên Validation RMSE.

In [8]:
# KHÔNG GIAN TÌM KIẾM
param_grid = {
    'rnn_type': ['LSTM', 'GRU'],
    'hidden_size': [64, 128],  # Thử nghiệm các kích thước chiều ẩn
    'num_layers': [1, 2],       # Stacked layers
    'bidirectional': [False, True],
    'dropout': [0.1, 0.3]       # Tỉ lệ ngắt kết nối
}

# Tạo tập hợp các phương án (Cartesian Product)
keys, values = zip(*param_grid.items())
combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
print(f"Tổng số cấu hình cần thử nghiệm: {len(combinations)}")


Tổng số cấu hình cần thử nghiệm: 32


In [9]:
# Thực thi Grid Search
results = []
best_rmse = float('inf')
best_params = None
best_model_state = None

print("=== START GRID SEARCH ===")
start_search = time.time()

for idx, params in enumerate(combinations):
    # Khởi tạo siêu mạng
    model_trial = DynamicRNN(
        rnn_type=params['rnn_type'],
        input_size=num_features,
        hidden_size=params['hidden_size'],
        num_layers=params['num_layers'],
        dropout=params['dropout'],
        bidirectional=params['bidirectional']
    )
    
    # Train Nhanh với số epochs phù hợp tránh chờ quá lâu
    final_val_mse = train_eval_model(model_trial, train_loader, val_loader, epochs=60, lr=1e-3, patience=7, verbose=False)
    
    # Lấy dự báo Val để tính RMSE thật (bằng cách Undo Log)
    val_preds_arr = predict(model_trial, X_v_t)
    val_metrics = regression_metrics(val_y.values, val_preds_arr)
    rmse_real = val_metrics['RMSE']
    
    results.append({'params': params, 'val_rmse': rmse_real, 'val_mae': val_metrics['MAE']})
    
    print(f"[{idx+1}/{len(combinations)}] {params} -> Val RMSE: {rmse_real:.4f}")
    
    # Ghi nhận Nhà Vô Địch
    if rmse_real < best_rmse:
        best_rmse = rmse_real
        best_params = params
        # (Chỉ reference tránh tốn thêm RAM)
        best_model_state = model_trial.state_dict()

print(f"\n=== GRID SEARCH HOÀN TẤT SAU {time.time()-start_search:.2f}s ===")
print("=> BEST PARAMS:", best_params)
print("=> BEST VAL RMSE:", best_rmse)


=== START GRID SEARCH ===
[1/32] {'rnn_type': 'LSTM', 'hidden_size': 64, 'num_layers': 1, 'bidirectional': False, 'dropout': 0.1} -> Val RMSE: 15.0207
[2/32] {'rnn_type': 'LSTM', 'hidden_size': 64, 'num_layers': 1, 'bidirectional': False, 'dropout': 0.3} -> Val RMSE: 15.2577
[3/32] {'rnn_type': 'LSTM', 'hidden_size': 64, 'num_layers': 1, 'bidirectional': True, 'dropout': 0.1} -> Val RMSE: 14.6557
[4/32] {'rnn_type': 'LSTM', 'hidden_size': 64, 'num_layers': 1, 'bidirectional': True, 'dropout': 0.3} -> Val RMSE: 14.6039
[5/32] {'rnn_type': 'LSTM', 'hidden_size': 64, 'num_layers': 2, 'bidirectional': False, 'dropout': 0.1} -> Val RMSE: 14.8442
[6/32] {'rnn_type': 'LSTM', 'hidden_size': 64, 'num_layers': 2, 'bidirectional': False, 'dropout': 0.3} -> Val RMSE: 14.9927
[7/32] {'rnn_type': 'LSTM', 'hidden_size': 64, 'num_layers': 2, 'bidirectional': True, 'dropout': 0.1} -> Val RMSE: 15.7858
[8/32] {'rnn_type': 'LSTM', 'hidden_size': 64, 'num_layers': 2, 'bidirectional': True, 'dropout': 0.3}

## 4. Kiểm tra Cấu hình Tốt nhất trên tập Test & Caching

Phục hồi trọng số (Weights) của mô hình Tối ưu nhất và tiến hành chạy dữ liệu cho toàn bộ các tập (Train/Val/Test). Kết quả sau đó được lưu trực tiếp dưới dạng `.pkl`.

In [10]:
# Re-instantiate The Best Model
best_dl = DynamicRNN(
    rnn_type=best_params['rnn_type'],
    input_size=num_features,
    hidden_size=best_params['hidden_size'],
    num_layers=best_params['num_layers'],
    dropout=best_params['dropout'],
    bidirectional=best_params['bidirectional']
)
best_dl.load_state_dict(best_model_state)
best_dl.to(device)

# Trích xuất Predictions cuối cùng
dl_preds_train = predict(best_dl, X_tr_t)
dl_preds_val = predict(best_dl, X_v_t)
dl_preds_test = predict(best_dl, X_te_t)

print("--- HIỆU SUẤT TRÊN TẬP TEST (BEST DL) ---")
test_eval = regression_metrics(test_y.values, dl_preds_test)
print(test_eval)

# Xuất ra file
dl_export = {
    "tuned_dl": {
        "train": dl_preds_train,
        "val": dl_preds_val,
        "test": dl_preds_test,
        "params": best_params
    }
}

dl_pickle_path = PRED_DIR / "tuned_dl_preds.pkl"
with open(dl_pickle_path, 'wb') as handle:
     pickle.dump(dl_export, handle, protocol=pickle.HIGHEST_PROTOCOL)
        

--- HIỆU SUẤT TRÊN TẬP TEST (BEST DL) ---
{'RMSE': 17.799221755740593, 'MAE': 12.172971494438977, 'MAPE': np.float64(30.414687842842504)}
